## Audit log — who did what, and when

Every UC permission grant, every table access, every cross-account read is recorded in the **Unity Catalog audit log** — the **forensic source of truth**. *"Did anyone query `silver.customers.email` outside compliance hours?"* is an audit-log question.

The **`system.access` schema** in the system catalog exposes audit data as **queryable tables**:

```sql
SELECT event_time, user_identity.email, action_name,
       request_params.full_name_arg AS object
FROM   system.access.audit
WHERE  event_time > current_timestamp() - INTERVAL 1 DAY
  AND  request_params.full_name_arg LIKE 'fintech_prod.silver.customers%'
ORDER BY event_time DESC;
```

- **`system.access.audit`** — every action: who, what, when, from where.
- **`system.access.table_lineage`** — which jobs / notebooks read which tables and wrote which tables, down to **column-level lineage**. This is what powers the **Lineage** tab in the UC UI.

**Two exam tells:**

- *"forensic audit — who read this table last Tuesday?"* → **`system.access.audit`**.
- *"track column-level read/write lineage"* → **`system.access.table_lineage`** (and the UC Lineage tab).

The audit log is queryable SQL, not just a UI — which means you can build alerts and dashboards on access patterns, the same way you'd monitor any other table.
